In [14]:
import json
from pathlib import Path

import requests

_API_URL = "https://api.bseindia.com/BseIndiaAPI/api/Corpforthresults/w"
_OUTPUT_PATH = Path(__file__).resolve().parent / "forth_results.json" if "__file__" in globals() else Path("forth_results.json")
_PAGE_URL = "https://www.bseindia.com/corporates/forth_results.html"

# BSE blocks generic Python user agents (Akamai WAF). Match other clients in this repo.
_HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) Chrome/124.0.0.0 Safari/537.36"
    ),
    "Accept": "application/json, text/plain, */*",
    "Accept-Language": "en-US,en;q=0.9",
    "Origin": "https://www.bseindia.com",
    "Referer": _PAGE_URL,
}

params = {
    "fromdate": "20250621",
    "todate": "20270621",
    "scripcode": "",
}

session = requests.Session()
session.headers.update(_HEADERS)

# Warm up cookies from the page before hitting the API.
session.get(_PAGE_URL, timeout=30)

response = session.get(_API_URL, params=params, timeout=30)
response.raise_for_status()

results = response.json()
# _OUTPUT_PATH.write_text(json.dumps(results, indent=2), encoding="utf-8")
# print(f"Saved {len(results)} records to {_OUTPUT_PATH.resolve()}")

In [15]:
from datetime import datetime, timedelta

latest = results[0]

scrip_code = latest["scrip_Code"]
short_name = latest["short_name"]
company_name = latest["Long_Name"]
meeting_date = latest["meeting_date"]
company_url = latest["URL"]

# Parse meeting_date
meeting_dt = datetime.strptime(meeting_date, "%d %b %Y")
# from_date = (meeting_dt - timedelta(days=2)).strftime("%Y%m%d")
# to_date = (meeting_dt + timedelta(days=7)).strftime("%Y%m%d")
from_date = 20260101
to_date = 20260622
subcategory = "Financial Results"

print(f"{company_name} ({scrip_code}) — meeting {meeting_date}")
print(from_date, to_date)

Vikas Lifecare Ltd (542655) — meeting 24 Jun 2026
20260101 20260622


Financial result

In [16]:
_ANN_SUBCAT_URL = "https://api.bseindia.com/BseIndiaAPI/api/AnnSubCategoryGetData/w"
_ANN_PAGE_URL = "https://www.bseindia.com/corporates/ann.html"


def fetch_ann_subcategory(
    scrip_code: str,
    subcategory: str,
    from_date: str,
    to_date: str,
    *,
    session: requests.Session | None = None,
) -> dict:
    """Fetch BSE corporate announcements for a scrip and result subcategory."""
    own_session = session is None
    if own_session:
        session = requests.Session()
        session.headers.update(_HEADERS)

    session.headers["Referer"] = _ANN_PAGE_URL
    session.get(_ANN_PAGE_URL, timeout=30)

    params = {
        "pageno": "1",
        "strCat": "Result",
        "strPrevDate": from_date,
        "strScrip": str(scrip_code).strip(),
        "strSearch": "P",
        "strToDate": to_date,
        "strType": "C",
        "subcategory": subcategory,
    }

    response = session.get(_ANN_SUBCAT_URL, params=params, timeout=30)
    response.raise_for_status()
    return response.json()


ann_results = fetch_ann_subcategory(
    scrip_code, subcategory, from_date, to_date, session=session
)
# Path("ann_subcategory_results.json").write_text(
#     json.dumps(ann_results, indent=2), encoding="utf-8"
# )
# print(f"Saved {len(ann_results.get('Table', []))} records")
ann_results

{'Table': [{'NEWSID': '3f89f404-0230-4d85-b05c-7d5dd8a9fb64',
   'SCRIP_CD': 542655,
   'XML_NAME': 'ANN_542655_3F89F404-0230-4D85-B05C-7D5DD8A9FB64',
   'NEWSSUB': ' Result  Of Board Meeting Held On Tuesday, April 21, 2026',
   'DT_TM': '2026-04-22T00:27:04.223',
   'NEWS_DT': '2026-04-22T00:27:04.223',
   'CRITICALNEWS': 0,
   'ANNOUNCEMENT_TYPE': 'A',
   'QUARTER_ID': None,
   'FILESTATUS': 'N    ',
   'ATTACHMENTNAME': 'cf53fa16-9790-470a-b291-f44d1e8bb2e5.pdf',
   'MORE': '',
   'HEADLINE': 'Result  of Board Meeting held on Tuesday, April 21, 2026',
   'CATEGORYNAME': 'Result',
   'OLD': 1,
   'RN': 1,
   'PDFFLAG': 1,
   'NSURL': 'https://www.bseindia.com/stock-share-price/vikas-lifecare-ltd/vikaslife/542655/',
   'SLONGNAME': 'Vikas Lifecare Ltd',
   'AGENDA_ID': 30,
   'News_submission_dt': '2026-04-22T00:27:04',
   'DissemDT': '2026-04-22T00:27:04.223',
   'TimeDiff': '00:00:00',
   'Fld_Attachsize': 4886819,
   'SUBCATNAME': 'Financial Results',
   'BSENewsid': None,
   'Inve

In [17]:
from urllib.parse import urljoin

latest_ann = max(ann_results["Table"], key=lambda row: row["NEWS_DT"])

nsurl = latest_ann["NSURL"]
attachment_name = latest_ann["ATTACHMENTNAME"].strip()

if attachment_name.lower().startswith(("http://", "https://")):
    pdf_url = attachment_name
else:
    pdf_url = urljoin(
        nsurl, f"/xml-data/corpfiling/AttachLive/{attachment_name.lstrip('/')}"
    )

session.headers["Referer"] = _ANN_PAGE_URL
pdf_response = session.get(pdf_url, timeout=60)
if pdf_response.status_code != 200 or not pdf_response.content.startswith(b"%PDF"):
    pdf_url = urljoin(
        nsurl, f"/xml-data/corpfiling/AttachHis/{attachment_name.lstrip('/')}"
    )
    pdf_response = session.get(pdf_url, timeout=60)
pdf_response.raise_for_status()

pdf_path = Path("downloads") / f"{scrip_code}_{attachment_name}"
pdf_path.parent.mkdir(parents=True, exist_ok=True)
pdf_path.write_bytes(pdf_response.content)

print(f"{latest_ann['HEADLINE']}")
print(f"URL: {pdf_url}")
print(f"Saved {len(pdf_response.content):,} bytes to {pdf_path.resolve()}")

Result  of Board Meeting held on Tuesday, April 21, 2026
URL: https://www.bseindia.com/xml-data/corpfiling/AttachHis/cf53fa16-9790-470a-b291-f44d1e8bb2e5.pdf
Saved 4,886,819 bytes to /Users/prairit/capital-nerve/source_discovery_layer/downloads/542655_cf53fa16-9790-470a-b291-f44d1e8bb2e5.pdf
